In [40]:
import pandas as pd 
df = pd.read_csv('./lending_club_2020_train.csv',low_memory=False )

In [50]:
df.shape

(1755295, 143)

In [43]:
df['loan_status'].value_counts()

loan_status
Fully Paid                                             898522
Current                                                618688
Charged Off                                            217366
Late (31-120 days)                                       9840
In Grace Period                                          6049
Late (16-30 days)                                        1620
Issued                                                   1258
Does not meet the credit policy. Status:Fully Paid       1223
Does not meet the credit policy. Status:Charged Off       460
Default                                                   268
Name: count, dtype: int64

In [44]:
#(원금 + 이자 ) / 대출받은 금액 
df['total_ratio'] = df['total_pymnt']/df['funded_amnt']

In [45]:
df['loan_status'].unique()

array(['Fully Paid', 'Current', 'Charged Off', 'Late (16-30 days)',
       'Late (31-120 days)', 'In Grace Period', 'Issued',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off', 'Default',
       nan], dtype=object)

In [46]:
df[df['loan_amnt']>df['funded_amnt']][['loan_amnt','funded_amnt']].head()

,loan_amnt,funded_amnt
864,35000.0,23200.0
976,25000.0,16125.0
1434,25000.0,15650.0
2421,8325.0,4225.0
3759,25000.0,23800.0


# 주요로 사용할 target 변수 
fully paid 란 원금을 모두 갚은 것을 의미합니다. 
금융권에서, 이자를 모두 갚은 뒤에, 원금을 상환할 수 있기에, 원금을 모두 상환했다는 것은 모든 원리금을 갚았다는 것을 의미합니다. 
>>추가 0821) 조기상환 시 모든 원리금을 갚지 않는 듯하다. 여기선 installment가 일정하게, 즉 원리금 균등방식을 사용하는듯.

- 계산에 사용한 변수
 1. 분자 : 지금까지 갚은 원금      / total_rec_prncp
 2. 분모 : 대출 시 빌린 총 금액    / funded_amnt

이를 예측 대상으로 둔다면, 대출 희망자가 앞으로 빌린 금액에서 돌려받을 기대 회수 비율을 예측 할 수 있습니다. 


In [47]:
df['target'] = df['total_rec_prncp']/df['funded_amnt']

## 해당 변수가 올바른지 확인하기 

1. Fully Paid 는 모두 1 이상인가?
   => 네 

In [39]:
temp1 = df.loc[(df['loan_status'] == 'Fully Paid') & (df['target'] < 0.999), :]
temp1['target']

389101    0.966326
Name: target, dtype: float64

In [32]:
df[df['target']>=1.0001]['target']

Series([], Name: target, dtype: float64)

In [35]:
temp2 = df[df['target']>=1.0001]
temp2=temp2[['loan_status','grade','total_ratio','target','int_rate','issue_d','loan_amnt','funded_amnt','total_rec_prncp','total_pymnt','installment','term','last_pymnt_d','last_pymnt_amnt','recoveries','collection_recovery_fee']]
temp2.head(30)

,loan_status,grade,total_ratio,target,int_rate,issue_d,loan_amnt,funded_amnt,total_rec_prncp,total_pymnt,installment,term,last_pymnt_d,last_pymnt_amnt,recoveries,collection_recovery_fee


In [9]:
df['target'].describe()

count    1.755294e+06
mean     6.793733e-01
std      3.734827e-01
min      0.000000e+00
25%      2.853232e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000012e+00
Name: target, dtype: float64

In [10]:
df.loc[(df['loan_status'] != 'Fully Paid') & (df['target'] > 0.99), :]['loan_status'].value_counts()

loan_status
Current                                               6652
Does not meet the credit policy. Status:Fully Paid    1223
Charged Off                                            111
Late (31-120 days)                                      98
In Grace Period                                         44
Late (16-30 days)                                       17
Default                                                 12
Name: count, dtype: int64

## 2. Charge off 인데, 1이상인거는 왜그럴까요? 
=> 이거는 어 거의 다 상환 ? 이거 뭐라했었지 

In [20]:
df['loan_status'].unique()

array(['Fully Paid', 'Current', 'Charged Off', 'Late (16-30 days)',
       'Late (31-120 days)', 'In Grace Period', 'Issued',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off', 'Default',
       nan], dtype=object)

In [60]:
fc=['Fully Paid','Current','Does not meet the credit policy. Status:Fully Paid']
temp21=df.loc[(~df['loan_status'].isin(fc)) & (df['target'] > 0.99999), :]
temp22=temp21[['loan_status','grade','total_ratio','target','int_rate','issue_d','loan_amnt','funded_amnt','total_rec_prncp','total_pymnt','installment','term','last_pymnt_d','last_pymnt_amnt','recoveries','collection_recovery_fee']]
temp22

,loan_status,grade,total_ratio,target,int_rate,issue_d,loan_amnt,funded_amnt,total_rec_prncp,total_pymnt,installment,term,last_pymnt_d,last_pymnt_amnt,recoveries,collection_recovery_fee
43205,Late (31-120 days),D,1.293354,1.0,16.99%,Mar-2017,13000.0,13000.0,13000.0,16813.600000,463.43,36 months,May-2020,30.37,0.00,0.0000
47114,In Grace Period,C,1.275640,1.0,14.99%,Feb-2017,7000.0,7000.0,7000.0,8929.479877,242.63,36 months,May-2020,258.74,0.00,0.0000
47745,Charged Off,B,1.143529,1.0,11.47%,Mar-2016,2550.0,2550.0,2550.0,2916.000000,84.06,36 months,Nov-2017,1397.06,9.11,1.6398
70918,In Grace Period,A,1.116998,1.0,7.24%,Apr-2017,16000.0,16000.0,16000.0,17871.970000,495.80,36 months,May-2020,507.05,0.00,0.0000
74869,In Grace Period,B,1.151052,1.0,9.44%,May-2017,4800.0,4800.0,4800.0,5525.050000,153.63,36 months,May-2020,137.70,0.00,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1625341,Charged Off,B,1.091395,1.0,11.99%,Oct-2017,7500.0,7500.0,7500.0,8185.460000,249.08,36 months,Sep-2018,5948.74,0.00,0.0000
1636663,Late (31-120 days),F,1.655580,1.0,24.99%,Nov-2015,27300.0,27300.0,27300.0,45197.330000,801.14,60 months,Apr-2020,3214.99,0.00,0.0000
1648237,In Grace Period,D,1.083560,1.0,18.62%,Aug-2019,9000.0,9000.0,9000.0,9752.040000,328.18,36 months,Apr-2020,4303.88,0.00,0.0000
1682301,In Grace Period,B,1.160003,1.0,9.93%,May-2017,6500.0,6500.0,6500.0,7540.018090,209.53,36 months,May-2020,419.59,0.00,0.0000


In [57]:
temp22[temp22['last_pymnt_amnt']>temp22['installment']+100]

,loan_status,grade,total_ratio,target,int_rate,issue_d,loan_amnt,funded_amnt,total_rec_prncp,total_pymnt,installment,term,last_pymnt_d,last_pymnt_amnt,recoveries,collection_recovery_fee
12133,In Grace Period,B,1.174819,0.998012,10.91%,May-2017,35000.0,35000.0,34930.43,41118.67,1144.37,36 months,Apr-2020,3375.67,0.00,0.0000
35097,Charged Off,E,1.543430,0.992665,21.97%,Mar-2016,20000.0,20000.0,19853.29,30868.59,552.04,60 months,Mar-2019,11100.00,287.45,51.7410
39361,Charged Off,F,1.665542,0.990296,24.08%,Oct-2014,10625.0,10625.0,10521.90,17696.38,306.16,60 months,Jul-2018,4210.00,0.00,0.0000
39402,Charged Off,D,1.269189,0.993421,16.99%,Mar-2015,7000.0,7000.0,6953.95,8884.32,249.54,36 months,Oct-2017,1644.00,10.27,1.8486
47745,Charged Off,B,1.143529,1.000000,11.47%,Mar-2016,2550.0,2550.0,2550.00,2916.00,84.06,36 months,Nov-2017,1397.06,9.11,1.6398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1699865,Charged Off,D,1.142155,0.999616,17.09%,May-2017,23000.0,23000.0,22991.16,26269.56,821.05,36 months,Apr-2018,18880.30,0.00,0.0000
1706759,Late (16-30 days),A,1.066684,0.993517,8.19%,Mar-2019,37200.0,37200.0,36958.85,39680.63,757.67,60 months,Mar-2020,32120.86,0.00,0.0000
1728394,Charged Off,G,1.476922,0.999007,30.99%,Dec-2016,3525.0,3525.0,3521.50,5206.15,151.56,36 months,Mar-2019,312.00,0.00,0.0000
1739617,Charged Off,B,1.162538,0.990667,10.49%,Jan-2015,20000.0,20000.0,19813.34,23250.77,649.96,36 months,Oct-2017,4394.00,31.24,5.6232


In [53]:
temp22['recoveries'].value_counts()

recoveries
0.00000      242
6.71000        1
17.41000       1
5.52000        1
4.73000        1
37.67000       1
17.48000       1
1.51000        1
19.21000       1
157.58000      1
42.25000       1
36.19999       1
26.23000       1
12.21000       1
0.14000        1
110.86000      1
7.62000        1
17.87000       1
31.24000       1
85.27000       1
124.54000      1
287.45000      1
50.00000       1
10.27000       1
9.11000        1
18.82000       1
38.73000       1
2.10000        1
80.78000       1
34.13000       1
30.92000       1
23.16000       1
2.35000        1
76.73000       1
3.90000        1
14.47000       1
139.70000      1
25.00000       1
9.23000        1
274.83000      1
70.74000       1
Name: count, dtype: int64

얘네는... 다 갚은거 같은데 심지어 조기상환도 많고... 

1. 다갚았는데 오류로 charged off로 분류되었거나 -> 트레이닝에 써야하는 데이터
2. 다 못갚았는데 (last_pymnt_d는 대출 기간 내 마지막 갚은 날짜) 대출 기간 끝난 후 전액상환 한 경우 -> 트레이닝에 써야하나? 애매한데..
3. 보증인이 갚았거나

2나 3일 때 그래도 기대수익으로 볼 수 있으니 트레이닝으로 써야하나?

In [49]:
temp2 = df.loc[(df['loan_status'] == 'Current') & (df['target'] >=0.99), :]
temp2=temp2[['loan_status','grade','total_ratio','target','int_rate','issue_d','loan_amnt','funded_amnt','total_rec_prncp','total_pymnt','installment','term','last_pymnt_d','last_pymnt_amnt','recoveries','collection_recovery_fee']]
temp2

,loan_status,grade,total_ratio,target,int_rate,issue_d,loan_amnt,funded_amnt,total_rec_prncp,total_pymnt,installment,term,last_pymnt_d,last_pymnt_amnt,recoveries,collection_recovery_fee
149,Current,B,1.303635,1.0,10.99%,May-2015,24000.0,24000.0,24000.0,31287.23,521.70,60 months,May-2020,521.58,0.0,0.0
158,Current,B,1.176448,1.0,10.91%,May-2017,6400.0,6400.0,6400.0,7529.27,209.26,36 months,May-2020,209.05,0.0,0.0
661,Current,A,1.064246,1.0,7.02%,Apr-2019,40000.0,40000.0,40000.0,42569.84,1235.45,36 months,May-2020,27760.04,0.0,0.0
793,Current,C,1.202070,1.0,15.02%,Jul-2018,25000.0,25000.0,25000.0,30051.76,866.88,36 months,May-2020,79.68,0.0,0.0
843,Current,B,1.187045,1.0,11.99%,Dec-2017,12000.0,12000.0,12000.0,14244.54,398.52,36 months,May-2020,3492.49,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754540,Current,C,1.179132,1.0,15.02%,Oct-2018,10000.0,10000.0,10000.0,11791.32,346.76,36 months,May-2020,0.73,0.0,0.0
1754618,Current,A,1.083055,1.0,5.32%,Aug-2017,6400.0,6400.0,6400.0,6931.55,192.74,36 months,May-2020,765.76,0.0,0.0
1754792,Current,A,1.034242,1.0,6.46%,Oct-2019,23000.0,23000.0,23000.0,23787.56,704.51,36 months,May-2020,6568.75,0.0,0.0
1755148,Current,B,1.241248,1.0,11.39%,May-2017,14500.0,14500.0,14500.0,17998.10,318.10,60 months,Jun-2020,0.00,0.0,0.0


In [13]:
temp2.shape

(111, 16)

In [14]:
temp2['grade'].value_counts()

grade
C    34
B    34
D    20
E    13
F     5
A     4
G     1
Name: count, dtype: int64

## 3. Current 1 이상 인거는 왜그럴까요? 
    모든 마지막 상환일이 2020 년도 인것으로 보아, 업데이트가 필요한 것으로 예상됩니다. 
    

0820추가) 이거 fully로 업데이트하면 위험할거같다...
0821추가) 아니다 괜찮다. 애초에 fully들도 조기상환이 엄청 많음. 그냥 업데이트 안한게 맞는듯

In [15]:
temp3 = df.loc[(df['loan_status'] == 'Current') & (df['target'] >=0.99), :]
temp4=temp3[['loan_status','total_ratio','target','int_rate','issue_d','funded_amnt','total_rec_prncp','total_pymnt','installment','term','last_pymnt_d','last_pymnt_amnt']]

In [16]:
temp4.head()

,loan_status,total_ratio,target,int_rate,issue_d,funded_amnt,total_rec_prncp,total_pymnt,installment,term,last_pymnt_d,last_pymnt_amnt
149,Current,1.303635,1.0,10.99%,May-2015,24000.0,24000.0,31287.23,521.70,60 months,May-2020,521.58
158,Current,1.176448,1.0,10.91%,May-2017,6400.0,6400.0,7529.27,209.26,36 months,May-2020,209.05
661,Current,1.064246,1.0,7.02%,Apr-2019,40000.0,40000.0,42569.84,1235.45,36 months,May-2020,27760.04
793,Current,1.202070,1.0,15.02%,Jul-2018,25000.0,25000.0,30051.76,866.88,36 months,May-2020,79.68
843,Current,1.187045,1.0,11.99%,Dec-2017,12000.0,12000.0,14244.54,398.52,36 months,May-2020,3492.49


In [17]:
temp4['last_pymnt_d'].unique()

array(['May-2020', 'Jun-2020', 'Apr-2020', 'Oct-2020', 'Mar-2020',
       'Jul-2020', 'Feb-2020'], dtype=object)

In [50]:
temp4['issue_d']=pd.to_datetime(temp4['issue_d'], format='%b-%Y')
temp4['last_pymnt_d']=pd.to_datetime(temp4['last_pymnt_d'], format='%b-%Y')
temp4['months_diff'] = ((temp4['last_pymnt_d'].dt.year - temp4['issue_d'].dt.year)*12 +
                       (temp4['last_pymnt_d'].dt.month - temp4['issue_d'].dt.month))
temp4.head()

/var/folders/vk/k2jbrnys5674kpkvxhf0q_8r0000gn/T/ipykernel_1491/3634703115.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp4['issue_d']=pd.to_datetime(temp4['issue_d'], format='%b-%Y')
/var/folders/vk/k2jbrnys5674kpkvxhf0q_8r0000gn/T/ipykernel_1491/3634703115.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp4['last_pymnt_d']=pd.to_datetime(temp4['last_pymnt_d'], format='%b-%Y')
/var/folders/vk/k2jbrnys5674kpkvxhf0q_8r0000gn/T/ipykernel_1491/3634703115.py:3: SettingWithCopyWarning: 
A value is

,loan_status,total_ratio,target,int_rate,issue_d,funded_amnt,total_rec_prncp,total_pymnt,installment,term,last_pymnt_d,last_pymnt_amnt,months_diff
149,Current,1.303635,1.0,10.99%,2015-05-01,24000.0,24000.0,31287.23,521.70,60 months,2020-05-01,521.58,60
158,Current,1.176448,1.0,10.91%,2017-05-01,6400.0,6400.0,7529.27,209.26,36 months,2020-05-01,209.05,36
661,Current,1.064246,1.0,7.02%,2019-04-01,40000.0,40000.0,42569.84,1235.45,36 months,2020-05-01,27760.04,13
793,Current,1.202070,1.0,15.02%,2018-07-01,25000.0,25000.0,30051.76,866.88,36 months,2020-05-01,79.68,22
843,Current,1.187045,1.0,11.99%,2017-12-01,12000.0,12000.0,14244.54,398.52,36 months,2020-05-01,3492.49,29


In [51]:
temp4['months_diff'].value_counts()

months_diff
36    944
60    284
35    257
33    179
34    174
     ... 
44     11
63      3
62      3
65      2
66      1
Name: count, Length: 65, dtype: int64

In [52]:
temp4.shape

(6652, 13)

In [53]:
temp3['last_pymnt_d'].unique()

array(['May-2020', 'Jun-2020', 'Apr-2020', 'Oct-2020', 'Mar-2020',
       'Jul-2020', 'Feb-2020'], dtype=object)

In [ ]:
temp

In [18]:
### 이제 loan_status 별로 